# Auditoria interna da Fase 3.8 ? execu??o manual

O treino e a compara??o Mirage da 3.8 j? terminaram. Este notebook preserva checkpoint e resultados, verifica as divis?es e audita formatos, resolu??es e erros apenas na sele??o interna. N?o infere a calibra??o, o teste OpenFake reservado ou o Mirage. O relat?rio da sele??o ? descritivo: essa parti??o j? escolheu o checkpoint e n?o mede generaliza??o independente. Execute as c?lulas em ordem; nenhuma inicia treino.


In [2]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

GPU: Tesla T4
Dependências disponíveis.


In [3]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_8', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 4.89 GB
/content livre: 247.19 GB


In [4]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [5]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'main'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase38-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('Notebooks/fase_3/fase3_8/fase3_8_aigi_piloto.py', 'Notebooks/fase_3/fase3_8/fase3_8_aigi_dados.py', 'Notebooks/fase_3/fase3_8/fase3_8_aigi_finetune.py', 'Notebooks/fase_3/fase3_8/avaliar_fase3_8_mirage.py', 'Notebooks/fase_3/fase3_5/fase3_5_dados.py', 'Notebooks/fase_3/fase3_7/fase3_7_dados.py', 'Notebooks/fase_3/fase3_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.8 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: fa606b96613385530be4768b050fb1bf15e2d819


In [6]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'Notebooks/fase_3/fase3_8/fase3_8_aigi_dados.py'
TRAIN = CODE_ROOT / 'Notebooks/fase_3/fase3_8/fase3_8_aigi_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

Código: /content/tcc_codigo/fa606b96613385530be4768b050fb1bf15e2d819


In [7]:
# 6. COPIA LOCAL: restaura 3.5, 3.7 e 3.8 do Drive; confere hashes.
OLD_DATA = CODE_ROOT / 'Notebooks/fase_3/fase3_5/fase3_5_dados.py'
PRIOR_DATA = CODE_ROOT / 'Notebooks/fase_3/fase3_7/fase3_7_dados.py'
run_live('copia local 3.5', [sys.executable, '-u', str(OLD_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local 3.7', [sys.executable, '-u', str(PRIOR_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local AIGI 3.8', [sys.executable, '-u', str(DATA), 'stage', '--root', str(ROOT)])


copia local 3.5: iniciando
Cópia/verificação: 100/3000
Cópia/verificação: 200/3000
Cópia/verificação: 300/3000
Cópia/verificação: 400/3000
Cópia/verificação: 500/3000
Cópia/verificação: 600/3000
Cópia/verificação: 700/3000
Cópia/verificação: 800/3000
Cópia/verificação: 900/3000
Cópia/verificação: 1000/3000
Cópia/verificação: 1100/3000
Cópia/verificação: 1200/3000
Cópia/verificação: 1300/3000
Cópia/verificação: 1400/3000
Cópia/verificação: 1500/3000
Cópia/verificação: 1600/3000
Cópia/verificação: 1700/3000
Cópia/verificação: 1800/3000
Cópia/verificação: 1900/3000
Cópia/verificação: 2000/3000
Cópia/verificação: 2100/3000
Cópia/verificação: 2200/3000
Cópia/verificação: 2300/3000
Cópia/verificação: 2400/3000
Cópia/verificação: 2500/3000
Cópia/verificação: 2600/3000
Cópia/verificação: 2700/3000
Cópia/verificação: 2800/3000
Cópia/verificação: 2900/3000
Cópia/verificação: 3000/3000
Dados internos locais prontos: /content/fase3_5_novos_v1
copia local 3.5: concluído em 0.1 min
copia local 3.7: 

In [8]:
# 7. AUDITORIA MANUAL: metadados de treino/sele??o e predi??es s? na sele??o.
AUDIT = CODE_ROOT / 'Notebooks/fase_3/fase3_8/auditoria_interna_fase3_8.py'
if not AUDIT.is_file():
    raise FileNotFoundError('Publique o script de auditoria na branch e execute novamente #4 e #5')
run_live('auditoria interna 3.8', [sys.executable, '-u', str(AUDIT),
                                  '--root', str(ROOT), '--code-root', str(CODE_ROOT)])


auditoria interna 3.8: iniciando
Partições conferidas: {'train': 6152, 'validation_selection': 1420, 'validation_calibration': 1420, 'test_reserved': 1947} | calibração e teste sem inferência
Metadados train: 500/6152
Metadados train: 1000/6152
Metadados train: 1500/6152
Metadados train: 2000/6152
Metadados train: 2500/6152
Metadados train: 3000/6152
Metadados train: 3500/6152
Metadados train: 4000/6152
Metadados train: 4500/6152
Metadados train: 5000/6152
Metadados train: 5500/6152
Metadados train: 6000/6152
Metadados train: 6152/6152
Metadados validation_selection: 500/1420
Metadados validation_selection: 1000/1420
Metadados validation_selection: 1420/1420
GPU: Tesla T4
Seleção interna: 104/1420 inferências
Seleção interna: 200/1420 inferências
Seleção interna: 304/1420 inferências
Seleção interna: 400/1420 inferências
Seleção interna: 504/1420 inferências
Seleção interna: 600/1420 inferências
Seleção interna: 704/1420 inferências
Seleção interna: 800/1420 inferências
Seleção interna

In [9]:
# 8. LEITURA SOMENTE-LEITURA: nenhum treino ou infer?ncia nesta c?lula.
import csv
report = ROOT / 'resultados/auditoria_interna_fase3_8_v1'
summary = report / 'summary.json'
if not summary.is_file():
    print('Auditoria ainda n?o conclu?da; execute #7 e confira o log.')
else:
    print(json.dumps(json.loads(summary.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
    with (report / 'segments_selection.csv').open(newline='', encoding='utf-8') as handle:
        segments = list(csv.DictReader(handle))
    print('Segmentos com 10+ imagens e maior taxa de erro (descri??o interna):')
    eligible = [row for row in segments if row['dimension'] in ('dataset_size_band', 'group') and int(row['n']) >= 10]
    for row in sorted(eligible, key=lambda item: (-float(item['error_rate']), -int(item['n'])))[:15]:
        print(row['dimension'], row['value'], 'classe=', row['label'],
              'erros=', row['errors'] + '/' + row['n'])


{
  "run_name": "fase3_8_aigi_pares256_v1",
  "counts": {
    "train": 6152,
    "validation_selection": 1420,
    "validation_calibration": 1420,
    "test_reserved": 1947
  },
  "selection_metrics_descriptive": {
    "n": 1420,
    "real": 630,
    "fake": 790,
    "accuracy": 0.9176056338028169,
    "balanced_accuracy": 0.9140546513964236,
    "precision": 0.9098660170523751,
    "recall_fake": 0.9455696202531646,
    "specificity_real": 0.8825396825396825,
    "f1": 0.9273743016759777,
    "tn": 556,
    "fp": 74,
    "fn": 43,
    "tp": 747,
    "auc": 0.9749367088607594
  },
  "temperature_frozen": 0.5229030847549438,
  "threshold_frozen": 0.30999999999999994,
  "checkpoint_sha256": "de4a11cbb733ad9088860d5f254cbb2a900f181cc1ccb28b2fd894cf4a37706a",
  "split_manifest_sha256": "22677a67d75f47b0c39257b016912ad21fabd5d5210c224a3010a702902b0106",
  "architecture_source_sha256": "3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b",
  "training_script_sha256": "84ae3564e9